# Gene annotation and gene profiling



## The workflow:

*   Bacterial gene prediction: prodigal
*   Generating the gene catalog: cd-hit
*   Building a bowtie2 database from the gene catalog: bowtie2
*   Mapping short reads against the gene catalog: bowtie2
*   Calculating the coverage for each sample

## Setup of the environment



In [9]:
# conda environment
import os,sys
root_dir = "/biodata/resources/day3_lab1"

## Check the installation was successful

In [10]:
! which prodigal
! which samtools
! which bowtie2
! which cd-hit


/opt/conda/bin/prodigal
/opt/conda/bin/samtools
/opt/conda/bin/bowtie2
/opt/conda/bin/cd-hit


## Predict bacterial genes on contigs

In [11]:
import os
root_dir = "/biodata/resources/day3_lab1"
# input folder
contigs_demo_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","contig_demo")
sample_id = "PSMB4MBK"
# input file: assembled contigs in fasta format
! ls -lh {contigs_demo_dir}/{sample_id}_contigs.fna
# output folder
gene_call_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call")
! mkdir -p {gene_call_dir}

-rw-rw-rw- 1 root root 58M Sep 29 15:13 /biodata/resources/day3_lab1/assembly_based_metagenomic_analysis/contig_demo/PSMB4MBK_contigs.fna


In [12]:
# pre-calculated gene call results
gene_call_demo_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_call_demo")

# check out the major output files
demo_output_gff = os.path.join(gene_call_demo_dir,sample_id+".prodigal.gff")
demo_output_faa = os.path.join(gene_call_demo_dir,sample_id+".prodigal.faa")
demo_output_fna = os.path.join(gene_call_demo_dir,sample_id+".prodigal.fna")

! head {demo_output_gff}

##gff-version  3
# Sequence Data: seqnum=1;seqlen=340;seqhdr="k105_1 flag=1 multi=4.0000 len=340"
# Model Data: version=Prodigal.v2.6.3;run_type=Metagenomic;model="1|Mycoplasma_pneumoniae_M129|B|40.0|4|0";gc_cont=40.00;transl_table=4;uses_sd=0
k105_1	Prodigal_v2.6.3	CDS	3	107	14.1	-	0	ID=1_1;partial=10;start_type=ATG;rbs_motif=None;rbs_spacer=None;gc_cont=0.429;conf=96.27;score=14.14;cscore=9.24;sscore=4.90;rscore=0.02;uscore=-0.52;tscore=5.40;
# Sequence Data: seqnum=2;seqlen=2417;seqhdr="k105_2 flag=1 multi=11.0000 len=2417"
# Model Data: version=Prodigal.v2.6.3;run_type=Metagenomic;model="11|Candidatus_Amoebophilus_asiaticus_5a2|B|35.0|11|0";gc_cont=35.00;transl_table=11;uses_sd=0
k105_2	Prodigal_v2.6.3	CDS	29	2416	285.4	-	0	ID=2_1;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.371;conf=99.99;score=285.37;cscore=282.16;sscore=3.22;rscore=0.00;uscore=0.00;tscore=3.22;
# Sequence Data: seqnum=3;seqlen=436;seqhdr="k105_3 flag=1 multi=7.0000 len=436"
# Model Data: v

In [8]:
# define the major out put files
output_gff = os.path.join(gene_call_dir,sample_id+".prodigal.gff")
output_faa = os.path.join(gene_call_dir,sample_id+".prodigal.faa")
output_fna = os.path.join(gene_call_dir,sample_id+".prodigal.fna")
# run gene prediction
! prodigal -p meta -i {contigs_demo_dir}/{sample_id}_contigs.fna -f gff -o {output_gff} -a {output_faa} -d {output_fna}  > /dev/null 2>&1 # take 7 min for run

## Overview of the output


In [14]:
! head {demo_output_faa}

>k105_1_1 # 3 # 107 # -1 # ID=1_1;partial=10;start_type=ATG;rbs_motif=None;rbs_spacer=None;gc_cont=0.429
MDELTDIYKRIEYLRNNGVKMKEIADRVDMAPSVL
>k105_2_1 # 29 # 2416 # -1 # ID=2_1;partial=01;start_type=Edge;rbs_motif=None;rbs_spacer=None;gc_cont=0.371
SITVRKLGRMPKIVKDPYIQASYKKIMGKPWYDLYSDEDLKYIEKMKEDPTLPNVIPSFS
NPEYYTYLGNTDWFSEIYDNTGITHSHNLSLSGASEKASYYIGMEYMQERGLLKINKDIM
DRYNFRSKVDFKVADWLTFGNNTSALYYTYKRPSSFYSWLFNRINDTNTLMTVKNPDGSW
TKEGAELIGSLSEGEAQTTELSLQSQFTMTLALIKNVLSIKADATARLGNRETEQWDSDM
NIPYKQGPNLADEYLGWVDMAQLAKERDYYTSVNAYIDFTKSFGKHQVSALAGFNQEYNS
HRYMRGEREELISSSLPSVELATGSARVREDNYEWATRGGFFRLNYIYNNKYIFEANGRY
DGSSRFPKNDRFGFFPSISAAWIISQEKFMHATQNWLNYAKLRISYGALGNQDVSYYEYI


### Task: count all complete genes

Incomplete genes are marked by the "partial" field in the header:

*   A complete gene: **partial=00**
*   A gene that is incomplete on the left side: **partial=10**
*   A gene that is incomplete on the right side: **partial=01**
*   A gene that is incomplete on both sides: **partial=11**

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> Calculate the number of complete genes and the total number of genes.
</div>


Hint:

*   **grep** the headers with **partial=00**
*   count the results using the **wc** commend




<details>
<summary><strong>🔎 Solution :</strong></summary>

```

! grep "partial=00" {output_fna} | wc -l
! grep ">" {output_fna} | wc -l

```

</details>


### Put your solution here [2 min]


In [ ]:
# your solution here:


## Generating the gene catalog

A gene catalog consist of all possible, **non-redundent** protein-coding genes

Assuming that we have assembled genes from multiple samples, we are going to generate a gene catalog from these files.

We will start with a merged file with all the **complete** genes from multiple samples

In [15]:
demo_gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog_demo")
merged_fa = os.path.join(demo_gene_catalog_dir,"merged_genes.fna")
! head {merged_fa}

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [16]:
coverage=90
identity=95
gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
! mkdir -p {gene_catalog_dir}
! cd-hit-est -i {merged_fa} -aS 0.{coverage} -aL 0.{coverage} -c 0.{identity} -M 0 -r 0 -B 0 -d 0 -o {gene_catalog_dir}/nr.fa -sc 1;

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 21:59:25
Command: cd-hit-est -i
         /biodata/resources/day3_lab1/assembly_based_metagenomic_analysis/gene_catalog_demo/merged_genes.fna
         -aS 0.90 -aL 0.90 -c 0.95 -M 0 -r 0 -B 0 -d 0 -o
         /biodata/resources/day3_lab1/assembly_based_metagenomic_analysis/gene_catalog/nr.fa
         -sc 1

Started: Tue Oct  7 05:12:24 2025
                            Output                              
----------------------------------------------------------------
total seq: 21584
longest and shortest : 12213 and 90
Total letters: 18729594
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 22M
Buffer          : 1 X 20M = 20M
Table           : 1 X 17M = 17M
Miscellaneous   : 0M
Total           : 59M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 647106500

comparing sequences from          0  to      21584
..........    10000  fini

### Overview of the result

There are two major output files:


1.   The representative sequence of each cluster [fasta format]
2.   A cluster file recording the cluster each gene originates from [.clsr]



In [17]:
! head {demo_gene_catalog_dir}/nr.fa

>CSM5MCXT_k105_1::1::76::309::-
ATGAAAGATATAGAAGTAAACGGCGCACATATAACAGATGAAAGTGCCGAGATTTTGAAA
CAGTGGCAAGTTAAGACGGAACCGGTTTCCGCTTGTTACATCGAAGTTATTGAGGACCTA
ATCGATTTCCTAATAGAGAAAGGAGATGAAAGTACACCAACAAATGAGGTGTTAAGAAGG
ATTCAATTATTACGTATGATGAAAAAAGACATCGAAAAGTTGTCTAATCCTTAA
>CSM5MCXT_k105_1::2::323::697::-
ATGAATACAAATAATCCTGATATTCTATTTTTCGTTAGACGTGAATACGGTGCACCTTCC
ATTGAATTAAGAGCATATAAGGTGGAGAAGGTAAACGAAGAATTTGCTTTCCTCGAACTT
GAACGTTTGCGGTTGGTTGTTTTCTCCGGTGATTTTCAGTCTGTATCACTTCATCACGAG
TACGGTAAAAACAACTGTCTGTATAATAGTGCCAATAATATACCGGATTTGATGAAAGAC


In [18]:
! head {demo_gene_catalog_dir}/nr.fa.clstr

>Cluster 0
0	117nt, >CSM5MCXT_k105_7934::2::331::447::-... *
1	117nt, >CSM5MCXT_k105_39322::2::174::290::+... at +/97.44%
2	117nt, >CSM5MCXT_k105_41278::23::23895::24011::-... at +/95.73%
3	117nt, >CSM5MCW6_k105_30839::49::69293::69409::-... at +/95.73%
>Cluster 1
0	315nt, >CSM5MCXT_k105_1156::2::349::663::+... *
1	315nt, >CSM5MCXT_k105_14525::2::349::663::+... at +/96.83%
2	315nt, >CSM5MCW6_k105_8476::1::62::376::-... at +/98.10%
>Cluster 2


### Task: count the number of resulting clusters

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many clusters are there?
</div>

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

# option 1: 
! tail {demo_gene_catalog_dir}/nr.fa.clstr

# option 2:
! grep ">Cluster" {demo_gene_call_dir}/nr.fa.clstr | wc -l

```

</details>

In [19]:
# your solution here:


## Create a bowtie2 database from the gene catalog



In [20]:
# build bowtie2 database. 1 min for demo
! bowtie2-build {gene_catalog_dir}/nr.fa {gene_catalog_dir}/nr.fa_bowtie2DB

Settings:
  Output files: "/biodata/resources/day3_lab1/assembly_based_metagenomic_analysis/gene_catalog/nr.fa_bowtie2DB.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 4 (one in 16)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 0
  Sizeofs: void*:8, int:4, long:8, size_t:8
Input files DNA, FASTA:
  /biodata/resources/day3_lab1/assembly_based_metagenomic_analysis/gene_catalog/nr.fa
Building a SMALL index
Reading reference sizes
  Time reading reference sizes: 00:00:00
Calculating joined length
Writing header
Reserving space for joined string
Joining reference sequences
  Time to join reference sequences: 00:00:00
bmax according to bmaxDivN setting: 3984285
Using parameters --bmax 298821

In [21]:
! ls -lh {demo_gene_call_dir}/nr.fa_bowtie2DB*

ls: cannot access '{demo_gene_call_dir}/nr.fa_bowtie2DB*': No such file or directory


# Profile the abundance of each gene in each sample

**Workflow**


1.   Map the reads against the bowtie2 database
2.   Calculate the coverage of each gene





In [28]:
import os

gene_catalog_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","gene_catalog")
mgx_reads_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_reads")
output_sam_dir = os.path.join(root_dir,"assembly_based_metagenomic_analysis","mgx_gf_mapping","raw")
! mkdir -p {output_sam_dir}

# define which sample you want to look at
sample_id="CSM5MCXT" 
# input reads
p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")


In [29]:
! bowtie2 -x {gene_catalog_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset
! ls {gene_catalog_dir}

10000 reads; of these:
  10000 (100.00%) were paired; of these:
    7259 (72.59%) aligned concordantly 0 times
    2476 (24.76%) aligned concordantly exactly 1 time
    265 (2.65%) aligned concordantly >1 times
    ----
    7259 pairs aligned concordantly 0 times; of these:
      295 (4.06%) aligned discordantly 1 time
    ----
    6964 pairs aligned 0 times concordantly or discordantly; of these:
      13928 mates make up the pairs; of these:
        13239 (95.05%) aligned 0 times
        570 (4.09%) aligned exactly 1 time
        119 (0.85%) aligned >1 times
33.80% overall alignment rate
nr.fa		       nr.fa_bowtie2DB.3.bt2	  nr.fa_bowtie2DB.rev.2.bt2
nr.fa_bowtie2DB.1.bt2  nr.fa_bowtie2DB.4.bt2	  nr.fa.clstr
nr.fa_bowtie2DB.2.bt2  nr.fa_bowtie2DB.rev.1.bt2


<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many reads are mapped to the gene catalog? 
</div>

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> How many reads are mapped from sample PSMB4MBK? 
</div>
 

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

# define which sample you want to look at
sample_id="PSMB4MBK"

# input reads
p1 = os.path.join(mgx_reads_dir,sample_id+"_subset_R1.fastq.gz")
p2 = os.path.join(mgx_reads_dir,sample_id+"_subset_R2.fastq.gz")

! bowtie2 -x {gene_catalog_dir}/nr.fa_bowtie2DB -1 {p1} -2 {p2} -S {output_sam_dir}/{sample_id}.sam # 22 min to finish on full / 3 min on subset


```

</details>

In [25]:
# Your solution here


### Profiling the coverage from the sam file

In [26]:
! which samtools
! samtools --version
! samtools coverage --help

/opt/conda/bin/samtools
samtools 1.22.1
Using htslib 1.22.1
Copyright (C) 2025 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             /opt/conda/conda-bld/samtools_1752527820713/_build_env/bin/aarch64-conda-linux-gnu-cc
    CPPFLAGS:       -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /opt/conda/include
    CFLAGS:         -Wall -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O3 -pipe -isystem /opt/conda/include -fdebug-prefix-map=/opt/conda/conda-bld/samtools_1752527820713/work=/usr/local/src/conda/samtools-1.22.1 -fdebug-prefix-map=/opt/conda=/usr/local/src/conda-prefix
    LDFLAGS:        -Wl,-O2 -Wl,--sort-common -Wl,--as-needed -Wl,-z,relro -Wl,-z,now -Wl,--allow-shlib-undefined -Wl,-rpath,/opt/conda/lib -Wl,-rpath-link,/opt/conda/lib -L/opt/conda/lib
    HTSDIR:         
    LIBS:           
    CURSES_LIB:     -ltinfow -lncursesw

HTSlib compilation details:
    Features:       build=configure libcurl=yes S3=

In [30]:
sample_id="CSM5MCXT" 
# sort the sam file
!samtools sort {output_sam_dir}/{sample_id}.sam -o {output_sam_dir}/{sample_id}.sorted_bam # 3 min
# generate the coverage file
!samtools coverage {output_sam_dir}/{sample_id}.sorted_bam > {output_sam_dir}/{sample_id}.coverage.txt

In [31]:
# check the output:
! ls -lh {output_sam_dir}
! head {output_sam_dir}/{sample_id}.coverage.txt

total 20M
-rw-rw-rw- 1 root root 1.1M Oct  7 05:22 CSM5MCXT.coverage.txt
-rw-rw-rw- 1 root root 6.4M Oct  7 05:22 CSM5MCXT.sam
-rw-rw-rw- 1 root root 1.5M Oct  7 05:22 CSM5MCXT.sorted_bam
-rw-r--r-- 1 root root 1.1M Oct  7 05:21 PSMB4MBK.coverage.txt
-rw-r--r-- 1 root root 5.8M Oct  7 05:21 PSMB4MBK.sam
-rw-r--r-- 1 root root 1.4M Oct  7 05:21 PSMB4MBK.sorted_bam
#rname	startpos	endpos	numreads	covbases	coverage	meandepth	meanbaseq	meanmapq
CSM5MCXT_k105_2::4::1725::2441::-	1	717	2	190	26.4993	0.281729	36.3	42
CSM5MCXT_k105_42::1::204::1901::+	1	1698	2	202	11.8963	0.118963	35.1	42
CSM5MCXT_k105_42::3::2862::4184::+	1	1323	2	202	15.2683	0.152683	36.3	42
CSM5MCXT_k105_77::4::4521::5486::+	1	966	2	148	15.3209	0.153209	34.3	42
CSM5MCXT_k105_82::1::206::1282::+	1	1077	1	87	8.07799	0.0807799	36.8	0
CSM5MCXT_k105_100::2::1999::3468::-	1	1470	2	202	13.7415	0.137415	36.8	42
CSM5MCXT_k105_119::1::169::918::-	1	750	1	101	13.4667	0.134667	36.8	42
CSM5MCXT_k105_119::3::1859::2515::-	1	657	2	182	27.